# Verified schedule switching for crosstalk identification in quantum error correction

This notebook reproduces every result and figure of the study with the code of the repository
[hamamh66/verified-schedule-switching](https://github.com/hamamh66/verified-schedule-switching)
(archived release v1.1.0, DOI [10.5281/zenodo.24409558](https://doi.org/10.5281/zenodo.24409558)).

| Experiment | Content | Approx. CPU time |
|---|---|---|
| E1 | Verified banks and protection–information frontier (color code, surface code) | 5 min |
| E2–E4 | Observation-model validation, finite-sample decisions, estimation and monitoring (color code) | 10 min |
| E5 | Protection under crosstalk drift: 3 scenarios, 5 policies, 10 replicates (surface code) | 10 min |
| E6 | Verification of transitions between units (color code, surface code) | 30 min |
| Figures | Figures 1–7 | 1 min |

Set `RUN_MODE` in the next cell: `"figures"` regenerates the figures from the released result files in about a minute; `"full"` reruns every experiment (about one hour on a Colab CPU). Each step prints its progress.

In [ ]:
# Configuration
RUN_MODE = "figures"          # "figures" (released results) or "full" (rerun all experiments)
SAVE_TO_DRIVE = True          # copy result JSON files and figures to Google Drive
DRIVE_DIR = "/content/drive/MyDrive/Outputs/Verified_Schedule_Switching"
REPO_URL = "https://github.com/hamamh66/verified-schedule-switching"
REPO_TAG = "v1.1.0"
WORK = "/content/verified-schedule-switching"   # working copy on the Colab local disk
print(f"Mode: {RUN_MODE} | tag: {REPO_TAG} | working directory: {WORK}")

## 1. Code and dependencies

In [ ]:
import os, subprocess, sys, time

def sh(cmd, cwd=None):
    """Run a shell command and stream its output line by line."""
    t0 = time.time()
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print(f"[done in {time.time()-t0:.0f} s, exit code {p.returncode}]", flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")

if not os.path.isdir(WORK):
    sh(f"git clone --depth 1 --branch {REPO_TAG} {REPO_URL} {WORK}")
else:
    print("Repository already present:", WORK)
sh("pip install -q -r requirements.txt", cwd=WORK)
import stim, pymatching
print("stim", stim.__version__, "| pymatching", pymatching.__version__)

## 2. E1 — verified banks and the protection–information frontier
For each radius of the verification set $E^{+}$, candidate units are admitted if crosstalk events do not lower the circuit-level fault distance; the identifiable couplings are then computed with the baseline noise projected out. E2–E4 reuse the design matrices computed here.

In [ ]:
if RUN_MODE == "full":
    for code_name in ["color", "surface"]:
        sh(f"python3 -u experiments/e1_bank_frontier.py {code_name} results", cwd=WORK)
else:
    print("Skipped (RUN_MODE = 'figures'): using released results/e1_*.json")

## 3. E2–E4 — observation model, decisions, estimation and monitoring (color code)

In [ ]:
if RUN_MODE == "full":
    sh("python3 -u experiments/e234_identification.py results 234", cwd=WORK)
else:
    print("Skipped (RUN_MODE = 'figures'): using released results/e2_color.json, e3_color.json, e4_color.json")

## 4. E5 — protection under crosstalk drift (surface code)
Ten replicates in two batches, then merged into `results/e5_surface.json`.

In [ ]:
if RUN_MODE == "full":
    sh("python3 -u experiments/e5_protection.py results 7,11,13,17,19", cwd=WORK)
    sh("python3 -u experiments/e5_protection.py results 23,29,31,37,41", cwd=WORK)
sh("python3 -u experiments/e5_merge.py results", cwd=WORK)

## 5. E6 — verification of transitions between units
Every ordered pair of verified units is tested with a switch after the first and after the second round.

In [ ]:
if RUN_MODE == "full":
    for code_name in ["color", "surface"]:
        sh(f"python3 -u experiments/e6_transitions.py {code_name} results", cwd=WORK)
else:
    print("Skipped (RUN_MODE = 'figures'): using released results/e6_*.json")

## 6. Figures

In [ ]:
sh("python3 -u figures/make_figures.py results figures", cwd=WORK)
sh("python3 -u figures/fig2_schematic.py figures/Fig2_probe.pdf", cwd=WORK)
sh("ls -l figures/*.pdf", cwd=WORK)

In [ ]:
# Display the figures inline (PDF converted to PNG)
sh("apt-get -qq install -y poppler-utils > /dev/null")
from IPython.display import Image, display
import glob
for pdf in sorted(glob.glob(f"{WORK}/figures/Fig*.pdf")):
    png = pdf[:-4]
    subprocess.run(["pdftoppm", "-png", "-r", "90", "-singlefile", pdf, png], check=True)
    print(os.path.basename(pdf)); display(Image(png + ".png"))

## 7. Summary tables

In [ ]:
import json
R = f"{WORK}/results"
print("E1 — verified units / identifiable couplings by radius of E+")
for c in ["color", "surface"]:
    r = json.load(open(f"{R}/e1_{c}.json"))
    print(f"  {c}: |U|={len(r['U'])}, reference unit alone identifies {r['single_ref_identifiable']}")
    for x in r["radii"]:
        f = r["frontier"][str(x)]
        print(f"    r={x}: units={len(f['bank'])}/{len(r['schedules'])}, identifiable={f['n_ident']}")
print("E6 — verified transitions")
for c in ["color", "surface"]:
    r = json.load(open(f"{R}/e6_{c}.json"))
    print(f"  {c}: " + ", ".join(f"r={k}: {v['verified_edges']}/{v['ordered_pairs']}" for k, v in r["transitions"].items()))
print("E5 — time-averaged logical failure rate per gate moment (mean over replicates)")
r = json.load(open(f"{R}/e5_surface.json"))
for sc in r["scenarios"]:
    print(f"  {sc}: " + ", ".join(f"{p} {r['summary'][sc][p]['total']['mean']:.2e}" for p in
          ["fixed-ref", "uniform-switching", "adaptive", "fixed-serialized", "oracle"]))

## 8. Save results and figures to Google Drive
Only the small result files and figures are written to Drive; the working copy stays on the Colab local disk.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True); os.makedirs(f"{DRIVE_DIR}/figures", exist_ok=True)
    sh(f"cp results/*.json '{DRIVE_DIR}/results/'", cwd=WORK)
    sh(f"cp figures/*.pdf '{DRIVE_DIR}/figures/'", cwd=WORK)
    print("Saved to", DRIVE_DIR)
else:
    print("Drive copy disabled")